# Trabalho Prático de IA - Notebook base de coleta de dados do ArXiv

**Disciplina:** Inteligência Artificial - PPGCC - FACOM/UFMS  
**Professor:** Bruno M. Nogueira  
**Semestre:** 2026/1

Este notebook é um **ponto de partida** para o trabalho prático. Ele mostra como **coletar artigos do ArXiv** filtrando por categorias e palavras-chave relacionadas ao seu tema de pesquisa, normalizar os metadados e salvar uma coleção em `JSONL`.

Você é **livre (e encorajado)** para modificar este código: trocar a API, ajustar os filtros, adicionar campos, etc. O notebook serve apenas para que você não perca tempo na parte mais mecânica do trabalho.

**Atenção:** a etapa de coleta deve ser justificada no relatório (escolha do tema, palavras-chave, categorias, janela temporal). Veja a Seção 3-(a) do enunciado e o exemplo lá fornecido.

> **A API do ArXiv é instável.** Não é incomum ver erros `HTTP 429` ("too many requests") ou `HTTP 503` ("service unavailable"), especialmente em queries longas ou em horários de pico. Este notebook implementa **retry com backoff exponencial e retomada por offset**, e o arquivo de saída preserva o progresso. Se a célula de coleta falhar, basta **rodar a mesma célula novamente** mais tarde, ela continua de onde parou. Não existe "chave de API" do ArXiv: o limite é por IP.

## 1. Instalação de dependências

Vamos usar a biblioteca [`arxiv`](https://pypi.org/project/arxiv/), um *wrapper* sobre a API oficial do ArXiv que já cuida de paginação e *rate limiting* básico. Também usamos `pandas` para inspeção tabular.

In [1]:
# Execute apenas uma vez, descomentando se necessário.
# !pip install arxiv pandas tqdm

In [2]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import arxiv
import pandas as pd
from tqdm import tqdm

## 2. Configuração da coleta (você edita aqui!)

Adapte os campos abaixo ao **seu** tema de pesquisa. O exemplo segue o caso do enunciado ("Detecção de fraudes em transações financeiras com aprendizado profundo").

**Dica para reduzir erros 429/503:** use `PAGE_SIZE = 50` (em vez de 100), evite horários de pico (manhã EUA) e mantenha as palavras-chave razoavelmente concisas. Queries muito longas (muitos `OR`) tendem a falhar mais.

In [3]:
# -----------------------------------------------------------------------------
# DEFINA AQUI O ESCOPO DA SUA COLEÇÃO
# -----------------------------------------------------------------------------

# Tema de pesquisa: processamento e otimização de consultas em bancos de grafos.
# (3ª rodada: a query da 2ª rodada exauriu em 903 artigos -- ainda abaixo do
# mínimo de 1000. Ampliamos mais uma vez palavras-chave, categorias e janela.)
KEYWORDS = [
    "graph database",
    "graph query optimization",
    "graph query processing",
    "property graph",
    "subgraph matching",
    "graph pattern matching",
    "SPARQL query optimization",
    "SPARQL query processing",
    "Cypher query language",
    "graph indexing",
    "distributed graph query processing",
    "graph processing system",
    "RDF query processing",
    "triple store",
    "graph traversal",
    "join order optimization",
    "query plan optimization",
    "graph data management",
    "graph store",
    "knowledge graph query",
    "knowledge graph storage",
    "graph database benchmark",
    "graph reachability query",
    "regular path query",
    "shortest path query",
    "in-memory graph database",
    "Gremlin query language",
    "graph OLTP",
]

# Categorias do ArXiv a considerar (lista em https://arxiv.org/category_taxonomy).
# cs.DB = bancos de dados (principal); cs.AI = grafos de conhecimento;
# cs.DC = processamento distribuído; cs.DS = algoritmos/estruturas de grafos;
# cs.IR = recuperação de informação; cs.LG = aprendizado de máquina (otimizadores
# aprendidos); cs.SE = engenharia de software (sistemas de bancos de grafos).
CATEGORIES = ["cs.DB", "cs.AI", "cs.DC", "cs.DS", "cs.IR", "cs.LG", "cs.SE"]

# Janela temporal (ano de submissão). Use None para não restringir.
# Ampliada para 2008 para incluir trabalhos fundacionais sobre RDF/SPARQL e
# modelos de property graph anteriores à popularização de sistemas como Neo4j.
YEAR_FROM = 2008
YEAR_TO = 2026

# Tamanho-alvo aproximado da coleção (entre 1000 e 5000, conforme enunciado).
TARGET_SIZE = 1500

# Tamanho de cada página retornada pela API.
# 50 costuma ser mais robusto contra HTTP 429/503 do que 100.
PAGE_SIZE = 50

# Caminho de saída da coleção bruta.
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUTPUT_DIR / "arxiv_raw.jsonl"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

print("Configuração carregada.")
print(" - Keywords  :", KEYWORDS)
print(" - Categories:", CATEGORIES)
print(" - Years     :", YEAR_FROM, "->", YEAR_TO)
print(" - Target    :", TARGET_SIZE)
print(" - PageSize  :", PAGE_SIZE)

Configuração carregada.
 - Keywords  : ['graph database', 'graph query optimization', 'graph query processing', 'property graph', 'subgraph matching', 'graph pattern matching', 'SPARQL query optimization', 'SPARQL query processing', 'Cypher query language', 'graph indexing', 'distributed graph query processing', 'graph processing system', 'RDF query processing', 'triple store', 'graph traversal', 'join order optimization', 'query plan optimization', 'graph data management', 'graph store', 'knowledge graph query', 'knowledge graph storage', 'graph database benchmark', 'graph reachability query', 'regular path query', 'shortest path query', 'in-memory graph database', 'Gremlin query language', 'graph OLTP']
 - Categories: ['cs.DB', 'cs.AI', 'cs.DC', 'cs.DS', 'cs.IR', 'cs.LG', 'cs.SE']
 - Years     : 2008 -> 2026
 - Target    : 1500
 - PageSize  : 50


## 3. Coleta paginada via API do ArXiv (com retry/backoff)

A função abaixo monta uma *query* combinando palavras-chave (com `OR`) e categorias (com `OR`), e itera sobre os resultados em ordem decrescente de data de submissão.

**Como ela trata erros HTTP 429 (rate limit) e 503 (serviço indisponível):**
- Configura o cliente do `arxiv` com `delay_seconds=5` e `num_retries=8` --- mais paciente que o padrão.
- Encapsula a iteração em um *loop externo* com **backoff exponencial**: 60s → 120s → 240s → 480s → 600s entre tentativas.
- Mantém um *offset* de onde paramos: se cair, retoma a partir do mesmo ponto (sem re-baixar tudo).
- Salva incrementalmente em `arxiv_raw.jsonl` e *deduplica* por `arxiv_id`. **Você pode rodar a célula de coleta várias vezes** --- o progresso é preservado.

**Se mesmo assim não funcionar:** espere 15–30 minutos e tente de novo (o seu IP pode estar temporariamente bloqueado), ou reduza `PAGE_SIZE` para 25 e/ou diminua o número de palavras-chave.

In [4]:
def build_query(keywords, categories):
    """Monta a string de query no formato esperado pela API do ArXiv."""
    kw_part = " OR ".join([f'all:"{k}"' for k in keywords]) if keywords else ""
    cat_part = " OR ".join([f"cat:{c}" for c in categories]) if categories else ""

    parts = [p for p in [f"({kw_part})" if kw_part else "",
                          f"({cat_part})" if cat_part else ""] if p]
    return " AND ".join(parts) if parts else "all:*"


QUERY = build_query(KEYWORDS, CATEGORIES)
print("Query final:\n", QUERY)

Query final:
 (all:"graph database" OR all:"graph query optimization" OR all:"graph query processing" OR all:"property graph" OR all:"subgraph matching" OR all:"graph pattern matching" OR all:"SPARQL query optimization" OR all:"SPARQL query processing" OR all:"Cypher query language" OR all:"graph indexing" OR all:"distributed graph query processing" OR all:"graph processing system" OR all:"RDF query processing" OR all:"triple store" OR all:"graph traversal" OR all:"join order optimization" OR all:"query plan optimization" OR all:"graph data management" OR all:"graph store" OR all:"knowledge graph query" OR all:"knowledge graph storage" OR all:"graph database benchmark" OR all:"graph reachability query" OR all:"regular path query" OR all:"shortest path query" OR all:"in-memory graph database" OR all:"Gremlin query language" OR all:"graph OLTP") AND (cat:cs.DB OR cat:cs.AI OR cat:cs.DC OR cat:cs.DS OR cat:cs.IR OR cat:cs.LG OR cat:cs.SE)


In [5]:
def already_collected_ids(path: Path) -> set:
    """Lê o arquivo .jsonl (se existir) e retorna os IDs já salvos."""
    if not path.exists():
        return set()
    ids = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                ids.add(json.loads(line)["arxiv_id"])
            except Exception:
                continue
    return ids


def collect_arxiv(query, target_size, page_size, year_from, year_to,
                   out_path: Path,
                   max_outer_attempts: int = 6,
                   initial_backoff_seconds: int = 60):
    """Coleta artigos do ArXiv, salvando incrementalmente em JSONL.

    Robusto contra erros HTTP 429/503 da API do ArXiv:
    - Retry interno do cliente `arxiv` (delay_seconds=5, num_retries=8).
    - Loop externo com backoff exponencial caso a iteração falhe.
    - Retoma do mesmo offset após cada falha, sem re-baixar páginas inteiras.
    """
    client = arxiv.Client(page_size=page_size, delay_seconds=5, num_retries=8)

    seen = already_collected_ids(out_path)
    print(f"Já coletados anteriormente: {len(seen)} artigos.")

    saved_this_run = 0
    offset = 0          # quantos resultados já consumimos da API
    outer_attempt = 0   # quantas vezes o loop externo tentou

    while len(seen) < target_size and outer_attempt < max_outer_attempts:
        try:
            search = arxiv.Search(
                query=query,
                max_results=target_size * 3,  # 3x para sobrar margem após filtros
                sort_by=arxiv.SortCriterion.SubmittedDate,
                sort_order=arxiv.SortOrder.Descending,
            )

            print(f"\nIniciando/retomando do offset={offset} "
                  f"(salvos={len(seen)}, meta={target_size}).")

            with open(out_path, "a", encoding="utf-8") as f:
                pbar = tqdm(initial=len(seen), total=target_size,
                            desc="coletando")
                for result in client.results(search, offset=offset):
                    offset += 1
                    year = result.published.year if result.published else None
                    if year_from is not None and (year is None or year < year_from):
                        continue
                    if year_to is not None and (year is None or year > year_to):
                        continue

                    arxiv_id = result.get_short_id().split("v")[0]
                    if arxiv_id in seen:
                        continue

                    record = {
                        "arxiv_id": arxiv_id,
                        "title": (result.title or "").strip(),
                        "abstract": (result.summary or "").strip().replace("\n", " "),
                        "authors": [a.name for a in result.authors],
                        "categories": list(result.categories or []),
                        "primary_category": result.primary_category,
                        "published": result.published.isoformat() if result.published else None,
                        "updated": result.updated.isoformat() if result.updated else None,
                        "doi": result.doi,
                        "pdf_url": result.pdf_url,
                        "entry_id": result.entry_id,
                    }
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    seen.add(arxiv_id)
                    saved_this_run += 1
                    pbar.update(1)
                    pbar.set_postfix(offset=offset)

                    if len(seen) >= target_size:
                        break
                pbar.close()

            # Se chegou aqui sem exception, terminou normalmente.
            break

        except Exception as e:
            outer_attempt += 1
            wait = min(initial_backoff_seconds * (2 ** (outer_attempt - 1)), 600)
            print(f"\n[aviso] coleta interrompida (tentativa {outer_attempt}/"
                  f"{max_outer_attempts}): {type(e).__name__}: {e}")
            print(f"[aviso] aguardando {wait}s antes de retomar do offset={offset}...")
            for _ in tqdm(range(wait), desc="backoff", leave=False):
                time.sleep(1)

    print(f"\nColeta finalizada. Total acumulado em {out_path}: {len(seen)} artigos.")
    if len(seen) < target_size:
        print(f"[atenção] Não atingiu a meta de {target_size} (parou em {len(seen)}).")
        print( "[atenção] Você pode rodar esta célula novamente para continuar de onde parou.")
    return len(seen)

In [6]:
n = collect_arxiv(
    query=QUERY,
    target_size=TARGET_SIZE,
    page_size=PAGE_SIZE,
    year_from=YEAR_FROM,
    year_to=YEAR_TO,
    out_path=RAW_PATH,
)
n

Já coletados anteriormente: 903 artigos.

Iniciando/retomando do offset=0 (salvos=903, meta=1500).


coletando:  60%|██████    | 903/1500 [00:00<?, ?it/s]

coletando:  60%|██████    | 904/1500 [00:03<39:39,  3.99s/it]

coletando:  60%|██████    | 904/1500 [00:03<39:39,  3.99s/it, offset=7]

coletando:  60%|██████    | 905/1500 [00:03<39:35,  3.99s/it, offset=37]

...

coletando:  73%|███████▎  | 1096/1500 [03:03<05:57,  1.13it/s, offset=1096]

coletando:  73%|███████▎  | 1097/1500 [03:03<05:56,  1.13it/s, offset=1097]

coletando:  73%|███████▎  | 1097/1500 [03:03<06:20,  1.06it/s, offset=1097]


Coleta finalizada. Total acumulado em ../data/arxiv_raw.jsonl: 1097 artigos.
[atenção] Não atingiu a meta de 1500 (parou em 1097).
[atenção] Você pode rodar esta célula novamente para continuar de onde parou.


1097

### O que fazer se a coleta continuar falhando

1. **Espere e tente de novo.** A API do ArXiv pode temporariamente bloquear o seu IP. Aguarde 15--30 min e rode a célula acima novamente --- ela vai retomar.
2. **Reduza a query.** Menos `OR` $\Rightarrow$ menos custo no servidor. Tente com 2--3 palavras-chave mais discriminativas.
3. **Reduza `PAGE_SIZE`** para 25 (mais requisições, cada uma mais leve).
4. **Aumente o `delay_seconds`** do `arxiv.Client` para 10 ou 15 segundos.
5. **Plano B: use o *bulk access* do ArXiv** (S3 / Kaggle dump) --- baixa em uma vez só, sem rate limit, ao custo de não ter os artigos mais recentes do dia.
6. **Plano C: troque a fonte** para Semantic Scholar, OpenAlex ou DBLP (todas listadas na Seção 5 do enunciado).

## 4. Normalização e deduplicação

A coleta bruta pode conter duplicatas (mesmo `arxiv_id` em diferentes versões) ou registros sem abstract. Geramos uma versão limpa em `corpus.jsonl`.

In [7]:
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


raw = load_jsonl(RAW_PATH)
print("Registros brutos:", len(raw))

# Deduplicar por arxiv_id (mantém o mais recente por 'updated').
df = pd.DataFrame(raw)
df["updated_dt"] = pd.to_datetime(df["updated"], errors="coerce")
df = df.sort_values("updated_dt").drop_duplicates("arxiv_id", keep="last")

# Remover registros sem título ou sem abstract.
df = df[df["title"].str.len() > 0]
df = df[df["abstract"].str.len() > 50]  # abstracts muito curtos costumam ser ruído

print("Após deduplicação e limpeza:", len(df))
df.head(3)

Registros brutos: 1097
Após deduplicação e limpeza: 1097


,arxiv_id,title,abstract,authors,categories,primary_category,published,updated,doi,pdf_url,entry_id,updated_dt
1096,0807.3908,A Distributed Process Infrastructure for a Dis...,The Resource Description Framework (RDF) is co...,[Marko A. Rodriguez],"[cs.AI, cs.DL]",cs.AI,2008-07-24T15:16:16+00:00,2008-07-24T15:16:16+00:00,NaN,https://arxiv.org/pdf/0807.3908v1,http://arxiv.org/abs/0807.3908v1,2008-07-24 15:16:16+00:00
1094,0901.3620,Enterprise model verification and validation: ...,This article presents a Verification and Valid...,"[Vincent Chapurlat, Bernard Kamsu Foguem, Fran...",[cs.SE],cs.SE,2009-01-23T08:42:21+00:00,2009-01-23T08:42:21+00:00,NaN,https://arxiv.org/pdf/0901.3620v1,http://arxiv.org/abs/0901.3620v1,2009-01-23 08:42:21+00:00
1095,0812.3788,Foundations of SPARQL Query Optimization,The SPARQL query language is a recent W3C stan...,"[Michael Schmidt, Michael Meier, Georg Lausen]",[cs.DB],cs.DB,2008-12-19T13:51:57+00:00,2009-01-26T12:52:36+00:00,NaN,https://arxiv.org/pdf/0812.3788v2,http://arxiv.org/abs/0812.3788v2,2009-01-26 12:52:36+00:00


## 5. Salvamento em `corpus.jsonl`

Este é o arquivo que você vai usar nas próximas etapas (BM25, KNN, etc.).

In [8]:
cols = ["arxiv_id", "title", "abstract", "authors", "categories",
         "primary_category", "published", "doi", "pdf_url"]

with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    for _, row in df[cols].iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"Corpus salvo em: {CORPUS_PATH} ({len(df)} documentos).")

Corpus salvo em: ../data/corpus.jsonl (1097 documentos).


## 6. Verificação rápida da coleção (*sanity checks*)

Antes de partir para a recuperação, dê uma olhada na distribuição temporal, por categoria, e em alguns exemplos. Erros mais comuns: filtros muito apertados (corpus minúsculo), filtros muito frouxos (corpus contaminado com áreas alheias).

In [9]:
df["year"] = pd.to_datetime(df["published"], errors="coerce").dt.year
print("Distribuição por ano:")
print(df["year"].value_counts().sort_index())
print("\nDistribuição por categoria primária:")
print(df["primary_category"].value_counts().head(10))

Distribuição por ano:
year
2008      2
2009      1
2010      5
2011      3
2012     17
2013     17
2014     25
2015     30
2016     33
2017     44
2018     60
2019     63
2020     75
2021     78
2022     76
2023    104
2024    104
2025    199
2026    161
Name: count, dtype: int64

Distribuição por categoria primária:
primary_category
cs.DB    465
cs.DC     98
cs.LG     96
cs.AI     90
cs.DS     87
cs.SE     48
cs.CL     47
cs.IR     46
cs.CR     24
cs.LO     21
Name: count, dtype: int64


In [10]:
# Exemplos aleatórios para inspeção manual.
for _, row in df.sample(min(3, len(df)), random_state=42).iterrows():
    print("=" * 80)
    print("ID      :", row["arxiv_id"])
    print("Title   :", row["title"])
    print("Cats    :", row["categories"])
    print("Year    :", row["year"])
    print("Abstract:", row["abstract"][:400], "...")

ID      : 1304.4910
Title   : A Junction Tree Framework for Undirected Graphical Model Selection
Cats    : ['stat.ML', 'cs.AI', 'cs.IT']
Year    : 2013
Abstract: An undirected graphical model is a joint probability distribution defined on an undirected graph G*, where the vertices in the graph index a collection of random variables and the edges encode conditional independence relationships among random variables. The undirected graphical model selection (UGMS) problem is to estimate the graph G* given observations drawn from the undirected graphical model ...
ID      : 2310.00673
Title   : Learning Type Inference for Enhanced Dataflow Analysis
Cats    : ['cs.LG', 'cs.CR']
Year    : 2023
Abstract: Statically analyzing dynamically-typed code is a challenging endeavor, as even seemingly trivial tasks such as determining the targets of procedure calls are non-trivial without knowing the types of objects at compile time. Addressing this challenge, gradual typing is increasingly added to dy